# Education based on conversation


- [ ] fix the regex to load the pkl better
- [ ] handle the 7 file failures (bad metadata processing)

Notes on the transcript types
- M01: life map
- N01: field notes
- TL01: time log
- F01: transcription (track)

In [36]:
%pip install gdown pandas numpy seaborn matplotlib PyPDF2 openpyxl

Note: you may need to restart the kernel to use updated packages.


In [37]:
# Import files using a zip folder instead of mounting drive for security

import gdown
import zipfile

file_id = '193NYD-P_bLDO24WYKsSSTugmkRZkAtkJ'       # file ID from share link
output_filename = 'raw_interview_transcripts.zip'

try:
    gdown.download(id=file_id, output=output_filename, quiet=False)
    print(f"File '{output_filename}' downloaded successfully.")
except Exception as e:
    print(f"An error occurred during download: {e}")

with zipfile.ZipFile('raw_interview_transcripts.zip', 'r') as zip_ref:
    zip_ref.extractall()


Downloading...
From (original): https://drive.google.com/uc?id=193NYD-P_bLDO24WYKsSSTugmkRZkAtkJ
From (redirected): https://drive.google.com/uc?id=193NYD-P_bLDO24WYKsSSTugmkRZkAtkJ&confirm=t&uuid=952a2308-9406-4184-9563-6e099f71a16e
To: /Users/rlay/work/26s-curric-803-report/raw_interview_transcripts.zip
100%|██████████| 121M/121M [00:04<00:00, 29.2MB/s] 


File 'raw_interview_transcripts.zip' downloaded successfully.


In [38]:
# datascience libs
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# file handling libs
import os
import PyPDF2
import re

In [39]:
# load pdf file names into a list, only including F01 (transcription notes)
pdf_dir = './individual_transcripts/'

pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf') and "F01" in f and "Viet" not in f]

# verify files loaded
print(f"Found {len(pdf_files)} PDF files:")
for pdf_file in pdf_files[:5]:
    print(pdf_file)


Found 204 PDF files:
./individual_transcripts/VAOHP0101_F01.pdf
./individual_transcripts/VAHF0002_F01.pdf
./individual_transcripts/VAOHP0111_F01.pdf
./individual_transcripts/VAOHP0069_F01.pdf
./individual_transcripts/VAHF0010_F01_Eng.pdf


In [40]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts all text content from a given PDF file.

    Args:
        pdf_path (str): The path to the PDF file.

    Returns:
        str: The extracted text content, or an empty string if an error occurs.
    """
    text_content = ''
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page_num in range(len(reader.pages)):
                page = reader.pages[page_num]
                text_content += page.extract_text() + '\n'
        return text_content
    except FileNotFoundError:
        print(f"Error: File not found at {pdf_path}")
        return ""
    except Exception as e:
        print(f"Error extracting text from '{pdf_path}': {e}")
        return ""

In [41]:
# verify files have contents
first_pdf_file = pdf_files[0]
if first_pdf_file:
    first_pdf_extracted_text = extract_text_from_pdf(first_pdf_file)
    if first_pdf_extracted_text:
        print(f"Successfully extracted text from '{first_pdf_file}' using the function.\nFirst 500 characters of text:\n")
        display(f"{first_pdf_extracted_text[:500]}...")
    else:
        print(f"Failed to extract text from '{first_pdf_file}' using the function.")
else:
    print("No PDF files found to test the function with.")


Successfully extracted text from './individual_transcripts/VAOHP0101_F01.pdf' using the function.
First 500 characters of text:



'Vietnamese American Oral History Project, UC Irvine   Narrator: MARY HOANG LONG Interviewer: Nina Mai Thi Long Date: November 13, 2012 Location: Westminster, California Sub-Collection: Vietnamese American Experience Course, Fall 2012 Length of Interview: 01:32:16 NL: Today is Tuesday, November 13, 2012. This is Nina Long with the Vietnamese American Oral History Project and I am interviewing Ms. Mary Long. We are at her home in Westminster, California.               NL: Hi Ms. Long              ...'

In [42]:
columns = [
    'narrator_name',
    'narrator_initials',
    'interviewer_name',
    'interviewer_initials',
    'date',
    'location',
    'interview_length',
    'sub-collection',
    'narrator_dialogue',
    'interviewer_dialogue'
]
df = pd.DataFrame(columns=columns)
print("Empty DataFrame created with specified columns:")
print(df.head())

Empty DataFrame created with specified columns:
Empty DataFrame
Columns: [narrator_name, narrator_initials, interviewer_name, interviewer_initials, date, location, interview_length, sub-collection, narrator_dialogue, interviewer_dialogue]
Index: []


In [43]:
def _remove_filename_from_text(text_content, file_name):
    """
    Removes the file name from the raw text content of the PDF
    Args:
        text_content (str): The raw text content extracted from a PDF.
        file_name (str): The name of the PDF file.

    Returns:
        str: The text content without the file name in the headers.
    """
    # trim file_name to remove directory and file extensions
    # r'/([^\/]+?)_F01' regex to select specifically the part between rawcontent/{target}_F01
    file_name = file_name.replace(f"{os.path.dirname(file_name)}/", '')
    file_name = file_name.replace('_F01.pdf', '')
    file_name = file_name.replace('_F01.docx.pdf', '')
    file_name = re.sub(r'/([^\/]+?)_F01', '', file_name)
    text_content = re.sub(fr"{file_name}\d+", "", text_content)
    return text_content
# test
display(_remove_filename_from_text(first_pdf_extracted_text, pdf_files[0]))
display(pdf_files[0])

'Vietnamese American Oral History Project, UC Irvine   Narrator: MARY HOANG LONG Interviewer: Nina Mai Thi Long Date: November 13, 2012 Location: Westminster, California Sub-Collection: Vietnamese American Experience Course, Fall 2012 Length of Interview: 01:32:16 NL: Today is Tuesday, November 13, 2012. This is Nina Long with the Vietnamese American Oral History Project and I am interviewing Ms. Mary Long. We are at her home in Westminster, California.               NL: Hi Ms. Long              MHL: Hi Nina.                 NL: What is your full name?             MHL: My full name is Mary Hoang Long.             NL: And how old are you right now?            MHL: I am 46 years old.               NL: And when is your date of birth?            MHL: My date of birth is April 6, 1966.             NL: And what is your place of birth?           MHL: I was born in Vietnam and in Saigon City.            NL: What are your parents\' names?           MHL: My father name is True Hoang and my mom\'

'./individual_transcripts/VAOHP0101_F01.pdf'

In [44]:
def _initialize_metadata():
    """
    Initializes the metadata dictionary with None values.
    """
    return {
        'interview_id': None,
        'narrator_name': None,
        'interviewer_name': None,
        'date': None,
        'location': None,
        'interview_length': None,
        'sub-collection': None,
        'narrator_initials': None,
        'interviewer_initials': None,
        'narrator_dialogue': None,
        'interviewer_dialogue': None
    }

In [45]:
def _get_metadata_patterns():
    """
    Defines and returns regular expression patterns for each metadata field.
    """
    return {
        'narrator_name': r"Narrator:([\s\S]*?)(?:Interviewer:|Date:|Location:|Sub-Collection:|Length of Interview:|$)",
        'interviewer_name': r"Interviewer:([\s\S]*?)(?:Date:|Location:|Sub-Collection:|Length of Interview:|$)",
        'date': r"Date:([\s\S]*?)(?:Location:|Sub-Collection:|Length of Interview:|$)",
        'location': r"Location:([\s\S]*?)(?:Sub-Collection:|Length of Interview:|$)",
        'interview_length': r"Length of Interview:\s+(\d{2}:\d{2}:\d{2})",
        'sub-collection': r"Sub-Collection:([\s\S]*?)(?:Length of Interview:|$)"
    }

In [46]:
def _extract_initial_fields(text_content, patterns):
    """
    Extracts initial metadata fields from the given text content using the provided patterns.
    """
    metadata = _initialize_metadata()
    for key, pattern in patterns.items():
        match = re.search(pattern, text_content, re.IGNORECASE | re.DOTALL)
        if match:
            metadata[key] = match.group(1).strip()
    return metadata

In [47]:
def _refine_name_field(name_string):
    """
    Refines narrator_name or interviewer_name by truncating extraneous text.
    """
    return name_string.lower().title()

In [48]:
def _clean_all_values(data):
    """
    Cleans up residual newlines or extra spaces in all extracted string values in a dictionary.
    """
    for key, value in data.items():
        if value and isinstance(value, str):
            data[key] = re.sub(r'\s+', ' ', value).strip()
    return data

In [49]:
def _extract_dialogue_section(text_content):
    """
    Extracts the dialogue section of the transcript.
    If not found, it tries to find the start of the first speaker turn.

    Args:
        text_content (str): The full text content of the PDF.

    Returns:
        str: The dialogue section, or an empty string if no clear dialogue start is found.
    """
    nl_marker_index = text_content.find('NL:')
    if nl_marker_index != -1:
        return text_content[nl_marker_index:].strip()
    else:
        # This assumes dialogue starts with a speaker's initials (e.g., NDM:, PCN:).
        speaker_initials_pattern = re.compile(r'\b[A-Z]{2,}:', re.DOTALL)
        match = speaker_initials_pattern.search(text_content)
        if match:
            # Start dialogue from the beginning of the first identified speaker initial
            return text_content[match.start():].strip()
        else:
            # If neither initials are found, we cannot reliably determine
            # the start of the dialogue, so return an empty string.
            return ""

In [50]:
def _extract_actual_speaker_initials(text_content):
    """
    Parses the dialogue section to identify the interviewer's initials
    (the first initial found) and the narrator's initials (the second distinct initial found).

    Args:
        text_content (str): The full text content of the PDF.

    Returns:
        tuple: A tuple containing (narrator_initials, interviewer_initials).
    """
    narrator_initials = None
    interviewer_initials = None

    dialogue_section_full = _extract_dialogue_section(text_content)

    if not dialogue_section_full:
        return (None, None)

    speaker_initials_pattern = re.compile(r'\b([A-Z]{2,5}):')
    matches = list(speaker_initials_pattern.finditer(dialogue_section_full))

    for match in matches:
        current_initials = match.group(1)

        if not interviewer_initials:
            interviewer_initials = current_initials
        elif current_initials != interviewer_initials:
            narrator_initials = current_initials
            break # Found both, so stop

    return (narrator_initials, interviewer_initials)

In [61]:
def _parse_speakers_dialogue(dialogue_text, narrator_initials, interviewer_initials):
    """
    Parses the dialogue text and attributes lines to narrator and interviewer
    based on their initials.

    Args:
        dialogue_text (str): The extracted dialogue section of the transcript.
        narrator_initials (str): The initials of the narrator.
        interviewer_initials (str): The initials of the interviewer.

    Returns:
        tuple: A tuple containing (narrator_dialogue_list, interviewer_dialogue_list),
               where each is a list of their respective dialogue segments.
    """
    narrator_dialogue_list = []
    interviewer_dialogue_list = []

    if not narrator_initials or not interviewer_initials:
        # If initials are missing, we cannot reliably parse.
        raise ValueError(f"failing due to bad initials {narrator_initials = }\t{interviewer_initials = }")

    # Clean non-breaking spaces from dialogue text
    dialogue_text = dialogue_text.replace('\xa0', ' ')

    narrator_init_esc = re.escape(narrator_initials)
    interviewer_init_esc = re.escape(interviewer_initials)

    # Regex to find speaker turns: (SpeakerInitials:)(DialogueText)
    # The non-greedy match (.*?) captures dialogue until the next speaker initial or end of string.
    # The lookahead `(?= ... |$)` ensures that the delimiter itself is not consumed by the match,
    # allowing subsequent matches to start correctly.
    speaker_turn_pattern = re.compile(
        rf"({interviewer_init_esc}:|{narrator_init_esc}:)(.*?)(?=(?:{interviewer_init_esc}:|{narrator_init_esc}:)|$)",
        re.MULTILINE
    )

    for match in speaker_turn_pattern.finditer(dialogue_text):
        speaker_id = match.group(1).strip() # e.g., "NL:" or "MHL:"
        dialogue_segment = match.group(2).strip()

        if speaker_id == f"{interviewer_initials}:": # Corrected comparison
            interviewer_dialogue_list.append(dialogue_segment)
        elif speaker_id == f"{narrator_initials}:": # Corrected comparison
            narrator_dialogue_list.append(dialogue_segment)

    return narrator_dialogue_list, interviewer_dialogue_list

In [62]:
def extract_interview_data_for_dataframe(text_content, file_name):
    """
    Extracts all interview-related data (metadata, initials, and dialogue) from the raw text
    of a PDF, suitable for populating a DataFrame.

    Args:
        text_content (str): The raw text content extracted from a PDF.

    Returns:
        dict: A dictionary containing all extracted and derived fields.
    """
    # 0. Pre-processing: Normalize whitespace at beginning, Remove header content
    processed_text_content = re.sub(r'\s+', ' ', text_content).strip()
    processed_text_content = _remove_filename_from_text(text_content, file_name)

    # 1. Extract initial metadata fields
    patterns = _get_metadata_patterns()
    metadata = _extract_initial_fields(processed_text_content, patterns)

    # Post-processing and refinement of metadata
    metadata['date'] = metadata['date']
    metadata['interview_length'] = metadata['interview_length']
    metadata['narrator_name'] = _refine_name_field(metadata['narrator_name'])
    metadata['interviewer_name'] = _refine_name_field(metadata['interviewer_name'])

    # 2. Derive initials using _extract_actual_speaker_initials
    actual_narrator_initials, actual_interviewer_initials = _extract_actual_speaker_initials(processed_text_content)
    metadata['narrator_initials'] = actual_narrator_initials
    metadata['interviewer_initials'] = actual_interviewer_initials

    # 3. Extract and parse dialogue
    dialogue_section = _extract_dialogue_section(processed_text_content)
    # Pass derived initials to the dialogue parser
    narrator_dialogue, interviewer_dialogue = _parse_speakers_dialogue(
        dialogue_section, metadata['narrator_initials'], metadata['interviewer_initials']
    )
    metadata['narrator_dialogue'] = narrator_dialogue
    metadata['interviewer_dialogue'] = interviewer_dialogue

    # Clean all string values in the final dictionary
    final_data = _clean_all_values(metadata)

    return final_data

In [63]:
def process_pdf_file(pdf_file, verbose=False):
  extracted_text = extract_text_from_pdf(pdf_file)
  extracted_interview_data = extract_interview_data_for_dataframe(extracted_text, pdf_file)
  if verbose:
    display(pdf_file)
    display("Extracted Interview Data:")
    for key, value in extracted_interview_data.items():
      display(f"  {key}: {value}")
  # Add the file name to the dictionary before returning
  match = re.search(r'(VA...\d+)', pdf_file)
  if match:
    extracted_interview_data['interview_id'] = match.group(1)
  else:
    extracted_interview_data['interview_id'] = np.nan
  return extracted_interview_data

# test to make sure it works
test = process_pdf_file(pdf_files[13])
for key, value in test.items():
  # use display on each key, value so that the dialogue doesn't take a bajillion
  # lines and output the entire transcript
  display(f"{key}: {value}")

'interview_id: VAOHP0034'

'narrator_name: Nhung Tran'

'interviewer_name: Carmen Tran'

'date: March 2, 2012'

'location: Garden Grove, California Sub‐Collection: Vietnamese American Experience Winter 2012 Length of Interview: 01:12:57 CT: My name is Carmen Tran and today is March 2, 2012. I am going to interview Nhung Tran and we are at his home in Garden Grove, California. This interview is for the Vietnamese American project at UC Irvine. I’d like if you can state your name, age, and where you live current. NT: My name is Nhung Tran and I am 58 years old and I live in the city of Garden Grove, California. CT: And where were you born? NT: I was born in Saigon, Vietnam. CT: Describe your home town where you grew up. NT: Saigon is the big city and I live in the big city close to the airport, and Saigon is a lot of people. And we are make the business in Saigon. CT: What kind of business do you have? NT: We have the Chinese uh we sell the uh...Chinese medicine. CT: Uh, what were your neighbors like? NT: My neighbors is they…they are Vietnamese…but they are business too so but they very nice peop

...

'sub-collection: None'

'narrator_initials: NT'

'interviewer_initials: CT'

"narrator_dialogue: ['My name is Nhung Tran and I am 58 years old and I live in the city of Garden Grove, California.', 'I was born in Saigon, Vietnam.', 'Saigon is the big city and I live in the big city close to the airport, and Saigon is a lot of people. And we are make the business in Saigon.', 'We have the Chinese uh we sell the uh...Chinese medicine.', 'My neighbors is they…they are Vietnamese…but they are business too so but they very nice people and very close with the neighbors.', 'My…my education is in high school is in uh..senior school', 'The school is..the..uh private school…so uh', 'We study is English, Vietnamese, History, uh Trig, and uhm Biology.', 'My parents, my father’s is the uh..Quach Tran and my mother’s name is the Diep Phong.', 'My parents were…very nice and they love me and they uh make the business very successful.', 'Uh…no', 'At home we speak in Chinese and in uh my neighbor and uh school we speak Vietnamese.', 'Can you say again?', 'The uh…because my parent

"interviewer_dialogue: ['My name is Carmen Tran and today is March 2, 2012. I am going to interview Nhung Tran and we are at his home in Garden Grove, California. This interview is for the Vietnamese American project at UC Irvine. I’d like if you can state your name, age, and where you live current.', 'And where were you born?', 'Describe your home town where you grew up.', 'What kind of business do you have?', 'Uh, what were your neighbors like?', 'Describe your schooling in Vietnam. What level of education did you have?', 'What was your school like?', 'Uhm, what did you learn? What did you study', 'What were your parent’s names? Describe them.', 'Uhm, what did you know most about your parents and grandparents when you were a child?', 'Uh what did you know about your family name? Are there stories about its history or its origin?', 'What languages do you speak? Do you speak a different language in different settings such as homes, school or work?', 'What memorable stories have your fa

In [65]:
# attempt to load all files into dataframe
all_interview_data = []
failed_interview_files = []
failed_interview_data = []
df_saved_file_path = 'interview_data.xlsx'
df = None

try:
  df = pd.read_excel(df_saved_file_path)
  display(f"loading successful: {df.head()}")
except:
  display(f"{df_saved_file_path} not found, loading interview data into new file")
  for i, pdf_file in enumerate(pdf_files):
      print(f"Processing file {i+1}/{len(pdf_files)}: {pdf_file}")
      try:
        extracted_data = process_pdf_file(pdf_file)
        all_interview_data.append(extracted_data)
      except Exception as e:
        print(f"Error processing {pdf_file}: {e}")
        failed_interview_files.append(pdf_file)

  df = pd.DataFrame(all_interview_data)

  # Reorder columns to place 'interview_id' at the first position
  if 'interview_id' in df.columns:
    cols = ['interview_id'] + [col for col in df.columns if col != 'interview_id']
    df = df[cols]

  print("\nDataFrame created successfully. First 5 rows:")
  display(df.head())
  df.to_excel('interview_data.xlsx')

'interview_data.xlsx not found, loading interview data into new file'

Processing file 1/204: ./individual_transcripts/VAOHP0101_F01.pdf
Processing file 2/204: ./individual_transcripts/VAHF0002_F01.pdf
Processing file 3/204: ./individual_transcripts/VAOHP0111_F01.pdf
Processing file 4/204: ./individual_transcripts/VAOHP0069_F01.pdf
Error processing ./individual_transcripts/VAOHP0069_F01.pdf: failing due to bad initials narrator_initials = None	interviewer_initials = None
Processing file 5/204: ./individual_transcripts/VAHF0010_F01_Eng.pdf
Processing file 6/204: ./individual_transcripts/VAOHP0100_F01_Eng.pdf
Processing file 7/204: ./individual_transcripts/VAOHP0215_F01.pdf
Processing file 8/204: ./individual_transcripts/VAOHP0074.2_F01_Eng.pdf
Processing file 9/204: ./individual_transcripts/VAOHP0050_F01_Eng.pdf
Processing file 10/204: ./individual_transcripts/VAOHP00377_F01.pdf
Processing file 11/204: ./individual_transcripts/VAOHP0368_F01 Track2.pdf
Processing file 12/204: ./individual_transcripts/VAOHP0142_F01_Eng.pdf
Processing file 13/204: ./individua

,interview_id,narrator_name,interviewer_name,date,location,interview_length,sub-collection,narrator_initials,interviewer_initials,narrator_dialogue,interviewer_dialogue
0,VAOHP0101,Mary Hoang Long,Nina Mai Thi Long,"November 13, 2012","Westminster, California",01:32:16,"Vietnamese American Experience Course, Fall 2012",MHL,NL,"[Hi Nina., My full name is Mary Hoang Long., I...","[Today is Tuesday, November 13, 2012. This is ..."
1,VAHF0002,Charlie Van Le,Pham Quang Tuan,"November 5, 2010","Westminster, California",00:12:13,Vietnamese American Heritage Foundation 500 Or...,CVL,PQT,"[Right now, I just graduated from the Universi...",[I’d like to ask you if you can state your nam...
2,VAOHP0111,Alex Thai Nguyen,Samantha Erica Takahashi,"February 10, 2013","Westminster, California",01:19:24,"Linda Vo Class Oral Histories, 2013",ATN,SET,"[Yes, my name is Alex Thai Nguyen. A-L-E-X Tha...","[Today is Sunday, February 10, 2013. This is S..."
3,VAHF0010,Nguyễn Thị Hạnh Nhơn (Nthn),Nancy Bui/Trieu Giang (Tg) Date Of Interview: ...,NaN,NaN,01:23:22,Vietnamese American Heritage Foundation 500 Or...,NTHN,TG,"[Yes, my dear . My name is Nguyễn thi Hanh Nho...",[Mrs Nguyễn Hạnh Nhơn . Can you let us know wh...
4,VAOHP0100,Thomas Toan Phan,Winty Thoumaked,"November 15, 2012","Westminster, California",01:06:50,"Vietnamese American Experience Course, Fall 2012",TP,WT,"[My name is Thomas Toan Phan., My date of birt...",[Hello my name is Winty Thoumaked and today is...


/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_11941/4160527589.py:31: UserWarning: Cell contents too long (37674), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_11941/4160527589.py:31: UserWarning: Cell contents too long (51886), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_11941/4160527589.py:31: UserWarning: Cell contents too long (58419), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_11941/4160527589.py:31: UserWarning: Cell contents too long (65672), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10t96b5wj2y6qr4g40000gn/T/ipykernel_11941/4160527589.py:31: UserWarning: Cell contents too long (62963), truncated to 32767 characters
  df.to_excel('interview_data.xlsx')
/var/folders/g2/6ym21yr10

In [66]:
display(df)

,interview_id,narrator_name,interviewer_name,date,location,interview_length,sub-collection,narrator_initials,interviewer_initials,narrator_dialogue,interviewer_dialogue
0,VAOHP0101,Mary Hoang Long,Nina Mai Thi Long,"November 13, 2012","Westminster, California",01:32:16,"Vietnamese American Experience Course, Fall 2012",MHL,NL,"[Hi Nina., My full name is Mary Hoang Long., I...","[Today is Tuesday, November 13, 2012. This is ..."
1,VAHF0002,Charlie Van Le,Pham Quang Tuan,"November 5, 2010","Westminster, California",00:12:13,Vietnamese American Heritage Foundation 500 Or...,CVL,PQT,"[Right now, I just graduated from the Universi...",[I’d like to ask you if you can state your nam...
2,VAOHP0111,Alex Thai Nguyen,Samantha Erica Takahashi,"February 10, 2013","Westminster, California",01:19:24,"Linda Vo Class Oral Histories, 2013",ATN,SET,"[Yes, my name is Alex Thai Nguyen. A-L-E-X Tha...","[Today is Sunday, February 10, 2013. This is S..."
3,VAHF0010,Nguyễn Thị Hạnh Nhơn (Nthn),Nancy Bui/Trieu Giang (Tg) Date Of Interview: ...,NaN,NaN,01:23:22,Vietnamese American Heritage Foundation 500 Or...,NTHN,TG,"[Yes, my dear . My name is Nguyễn thi Hanh Nho...",[Mrs Nguyễn Hạnh Nhơn . Can you let us know wh...
4,VAOHP0100,Thomas Toan Phan,Winty Thoumaked,"November 15, 2012","Westminster, California",01:06:50,"Vietnamese American Experience Course, Fall 2012",TP,WT,"[My name is Thomas Toan Phan., My date of birt...",[Hello my name is Winty Thoumaked and today is...
...,...,...,...,...,...,...,...,...,...,...,...
181,VAHF0007,Andrew Tuan Pham (Aka Pham Quan Tuan),Roger Minh Le,"November 5, 2010","Westminster, California",00:21:12,Vietnamese American Heritage Foundation 500 Or...,PQT,RL,[My name is Andrew Tuan Pham. But I also go by...,"[Today is November 5, 2010. We are here in Wes..."
182,VAOHP0114,Steve Pham,David Liu,"February 23, 2013","Garden Grove, California",01:52:53,Linda Vo Class Oral Histories 2013,SP,DL,"[Steve Pham, 8/18/1960, Saigon, Vietnam, My fa...",[This is David Liu with the Vietnamese America...
183,VAOHP0084,Keith Ky Thanh Van,Thuý Võ Đặng,"August 8, 2012","Huntington Beach, California",01:36:36,Thuy Vo Dang Oral Histories,KV,TVD,"[My\t\r name\t\r is\t\r Keith\t\r Van,\t\r...",[First\t\r please\t\r introduce\t\r your\t\...
184,VAOHP0079,Kiên Tâm Nguyễn,Thúy Võ Đặng,"May 22, 2012","Fountain Valley, California Sub-­‐Collection: ...",NaN,NaN,KTN,TVD,[Ok.\t\r \t\r My\t\r name\t\r is\t\r Kien...,"[First\t\r of\t\r all,\t\r could\t\r you\t..."


In [67]:
display(failed_interview_files)

['./individual_transcripts/VAOHP0069_F01.pdf',
 './individual_transcripts/VAOHP0002_F01.pdf',
 './individual_transcripts/VAOHP0359_F01_Eng.pdf',
 './individual_transcripts/VAOHP0068_F01.pdf',
 './individual_transcripts/VAOHP0035_F01.pdf',
 './individual_transcripts/VAOHP0353_F01.pdf',
 './individual_transcripts/VAOHP0137_F01_Eng.pdf',
 './individual_transcripts/VAOHP0097_F01.pdf',
 './individual_transcripts/VAOHP0036_F01.pdf',
 './individual_transcripts/VAOHP0010_F01.pdf',
 './individual_transcripts/VAOHP0376_F01.pdf',
 './individual_transcripts/VAOHP0349_F01.pdf',
 './individual_transcripts/VAOHP0096_F01.pdf',
 './individual_transcripts/VAOHP0363_F01.pdf',
 './individual_transcripts/VAOHP0040.1_F01.pdf',
 './individual_transcripts/VAOHP0167_F01.pdf',
 './individual_transcripts/VAOHP0091_F01.pdf',
 './individual_transcripts/VAOHP0040.2_F01.pdf']

- VAOHP0097_F01.pdf: failed because missing `:` for interviewer
- VAOHP0010_F01.pdf: failed because missing `er` for interviewer
- VAOHP0002_F01.pdf: failed because missing `er` for interviewer
- VAOHP0036_F01.pdf: failed because extra whitespace `Inter viewer`
- VAOHP0167_F01.pdf: failed??
- VAOHP0035_F01.pdf: failed because missing `:` for interviewer
- VAOHP0376_F01.pdf: failed??

In [58]:
empty_dialogues_narrator = (df["narrator_dialogue"].isnull() | (df["narrator_dialogue"] == "")).sum()
print(f"Number of empty rows in 'narrator_dialogue': {empty_dialogues_narrator}")
empty_dialogues_interviewer = (df["interviewer_dialogue"].isnull() | (df["interviewer_dialogue"] == "")).sum()
print(f"Number of empty rows in 'interviewer_dialogue': {empty_dialogues_interviewer}")

Number of empty rows in 'narrator_dialogue': 0
Number of empty rows in 'interviewer_dialogue': 0
